In [1]:
import torch
from encoders import LatencyEncoder, PoissonEncoder

def _assert_binary_float32(x: torch.Tensor):
    assert x.dtype == torch.float32, f"dtype must be float32, got {x.dtype}"
    u = torch.unique(x.detach().cpu())
    bad = [v for v in u.tolist() if v not in (0.0, 1.0)]
    assert len(bad) == 0, f"tensor must be binary (0/1). Found: {bad[:10]}"

def _run(name, fn):
    try:
        fn()
        print(f"✅ {name}")
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}")
        raise

In [2]:
def test_latency_single_28x28_auto_shape_and_counts():
    T = 10
    enc = LatencyEncoder(time=T, out_format="auto")
    img = torch.zeros(28, 28, dtype=torch.float32)

    spk = enc(img)
    assert spk.shape == (T, 1, 784), f"got {spk.shape}"
    _assert_binary_float32(spk)

    # Для нулевой картинки t_fire = T-1 => все импульсы только на последнем шаге
    assert float(spk[:-1].sum()) == 0.0
    assert float(spk[-1].sum()) == 784.0

    # Ровно 1 импульс на пиксель за всё время
    per_pixel = spk.sum(dim=0)  # [1,784]
    assert torch.all(per_pixel == 1.0)

def test_latency_batch_Bx28x28_auto_shape():
    T = 11
    B = 3
    enc = LatencyEncoder(time=T, out_format="auto")
    img = torch.zeros(B, 28, 28, dtype=torch.float32)

    spk = enc(img)
    assert spk.shape == (T, B, 1, 784), f"got {spk.shape}"
    _assert_binary_float32(spk)

    spk_tbn = spk.view(T, B, 784)
    per_pixel = spk_tbn.sum(dim=0)  # [B,784]
    assert torch.all(per_pixel == 1.0)

def test_latency_mapping_extremes_TBN():
    T = 13
    enc = LatencyEncoder(time=T, out_format="TBN")
    img = torch.zeros(28, 28, dtype=torch.float32)
    img[0, 0] = 1.0   # должен быть t=0
    img[0, 1] = 0.0   # должен быть t=T-1
    img[0, 2] = 0.5   # t=floor(0.5*(T-1))

    spk = enc(img)  # [T,1,784]
    assert spk.shape == (T, 1, 784)
    _assert_binary_float32(spk)

    n0, n1, n2 = 0, 1, 2

    assert spk[0, 0, n0].item() == 1.0
    assert spk[1:, 0, n0].sum().item() == 0.0

    assert spk[T-1, 0, n1].item() == 1.0
    assert spk[:T-1, 0, n1].sum().item() == 0.0

    expected_t = int(torch.floor(torch.tensor((1.0 - 0.5) * (T - 1))).item())
    assert spk[expected_t, 0, n2].item() == 1.0

def test_latency_uint8_255_goes_to_first_step():
    T = 8
    enc = LatencyEncoder(time=T, out_format="auto")
    img = torch.full((28, 28), 255, dtype=torch.uint8)

    spk = enc(img)
    assert spk.shape == (T, 1, 784)
    _assert_binary_float32(spk)

    assert float(spk[0].sum()) == 784.0
    assert float(spk[1:].sum()) == 0.0


_run("Latency: single [28,28] -> shape + one-spike-per-pixel", test_latency_single_28x28_auto_shape_and_counts)
_run("Latency: batch [B,28,28] -> shape + one-spike-per-pixel", test_latency_batch_Bx28x28_auto_shape)
_run("Latency: mapping extremes (1.0->t0, 0.0->tT-1, 0.5->mid)", test_latency_mapping_extremes_TBN)
_run("Latency: uint8 normalization (255 -> t=0)", test_latency_uint8_255_goes_to_first_step)

✅ Latency: single [28,28] -> shape + one-spike-per-pixel
✅ Latency: batch [B,28,28] -> shape + one-spike-per-pixel
✅ Latency: mapping extremes (1.0->t0, 0.0->tT-1, 0.5->mid)
✅ Latency: uint8 normalization (255 -> t=0)


In [3]:
import torch
from encoders import LatencyEncoder, PoissonEncoder

def _run(name, fn):
    try:
        fn()
        print(f"✅ {name}")
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}")
        raise

def test_same_image_same_encoding_latency():
    T = 20
    enc = LatencyEncoder(time=T, out_format="auto")
    img = torch.rand(28, 28)  # одно и то же изображение

    spk1 = enc(img)
    spk2 = enc(img)

    assert torch.equal(spk1, spk2), "LatencyEncoder должен кодировать один и тот же вход одинаково"
    assert spk1.shape == spk2.shape

def test_same_image_same_encoding_poisson_deterministic():
    T = 20
    enc = PoissonEncoder(T=T, rate_scale=0.5, base_seed=123, deterministic=True, out_format="auto")
    img = torch.rand(28, 28)

    spk1 = enc(img)
    spk2 = enc(img)

    assert torch.equal(spk1, spk2), "PoissonEncoder при deterministic=True должен кодировать одинаково"
    assert spk1.shape == spk2.shape

def test_same_image_not_always_same_poisson_stochastic():
    T = 20
    enc = PoissonEncoder(T=T, rate_scale=0.5, deterministic=False, out_format="auto")
    img = torch.rand(28, 28)

    spk1 = enc(img)
    spk2 = enc(img)

    # В стохастическом режиме обычно будут отличаться.
    # Если вдруг совпали (редко), повторим несколько раз.
    if torch.equal(spk1, spk2):
        same_count = 1
        for _ in range(5):
            if torch.equal(enc(img), enc(img)):
                same_count += 1
        assert same_count < 3, (
            "PoissonEncoder при deterministic=False неожиданно часто даёт одинаковый результат; "
            "проверь, что deterministic действительно выключен."
        )

_run("Same image -> same encoding (Latency)", test_same_image_same_encoding_latency)
_run("Same image -> same encoding (Poisson deterministic)", test_same_image_same_encoding_poisson_deterministic)
_run("Same image -> usually different (Poisson stochastic)", test_same_image_not_always_same_poisson_stochastic)

✅ Same image -> same encoding (Latency)
✅ Same image -> same encoding (Poisson deterministic)
✅ Same image -> usually different (Poisson stochastic)


In [4]:
import torch
from encoders import LatencyEncoder

def _run(name, fn):
    try:
        fn()
        print(f"✅ {name}")
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}")
        raise

def test_latency_xmin_filters_background_all_zeros():
    """
    Полностью нулевая картинка: при x_min>0 импульсов быть не должно.
    """
    T = 20
    enc = LatencyEncoder(time=T, out_format="TBN", x_min=0.05)
    img = torch.zeros(28, 28, dtype=torch.float32)

    spk = enc(img)  # [T,1,784]
    assert spk.shape == (T, 1, 784)
    assert float(spk.sum()) == 0.0, "для нулевой картинки и x_min>0 импульсов быть не должно"

def test_latency_xmin_keeps_only_active_pixels_and_one_spike_each():
    """
    Делаем картинку, где активны только 2 пикселя.
    Проверяем: (а) импульсы только по ним, (б) ровно один импульс на активный пиксель.
    """
    T = 30
    x_min = 0.1
    enc = LatencyEncoder(time=T, out_format="TBN", x_min=x_min)

    img = torch.zeros(28, 28, dtype=torch.float32)
    img[0, 0] = 1.0   # активный
    img[0, 1] = 0.5   # активный
    img[0, 2] = 0.05  # ниже порога -> НЕ активный

    spk = enc(img)  # [T,1,784]
    assert spk.shape == (T, 1, 784)

    # всего должно быть 2 импульса (по одному на активный пиксель)
    assert float(spk.sum()) == 2.0

    # индексы пикселей (0,0)->0, (0,1)->1, (0,2)->2
    n0, n1, n2 = 0, 1, 2

    # по активным — ровно один спайк
    assert float(spk[:, 0, n0].sum()) == 1.0
    assert float(spk[:, 0, n1].sum()) == 1.0

    # по неактивному (ниже x_min) — ноль
    assert float(spk[:, 0, n2].sum()) == 0.0

def test_latency_xmin_reduces_last_timestep_burst_on_sparse_digits_like_input():
    """
    Идея: без x_min фон генерирует много спайков на t=T-1.
    С x_min>0 фон отфильтрован => спайков на последнем шаге значительно меньше.
    (Мы не сравниваем с "старым" encoder, просто проверяем, что последний шаг не доминирует
     из-за фоновых пикселей, на простой искусственной картинке.)
    """
    T = 50
    enc = LatencyEncoder(time=T, out_format="TBN", x_min=0.1)

    # имитация "цифры": активна только небольшая часть пикселей
    img = torch.zeros(28, 28, dtype=torch.float32)
    img[10:13, 10:13] = 0.8  # 9 пикселей
    img[14, 14] = 1.0        # ещё один пиксель

    spk = enc(img)  # [T,1,784]
    assert float(spk.sum()) == 10.0  # 10 активных пикселей -> 10 спайков всего

    per_t = spk[:, 0, :].sum(dim=1)  # [T]
    # последний шаг не должен содержать “массу” спайков от фона — максимум 10
    assert float(per_t[-1]) <= 10.0
    # и вообще максимум по времени не должен быть огромным
    assert float(per_t.max()) <= 10.0

_run("Latency x_min: zero image -> zero spikes", test_latency_xmin_filters_background_all_zeros)
_run("Latency x_min: keeps only active pixels + one spike each", test_latency_xmin_keeps_only_active_pixels_and_one_spike_each)
_run("Latency x_min: no last-step background burst on sparse input", test_latency_xmin_reduces_last_timestep_burst_on_sparse_digits_like_input)

✅ Latency x_min: zero image -> zero spikes
✅ Latency x_min: keeps only active pixels + one spike each
✅ Latency x_min: no last-step background burst on sparse input
